# rfdetr

> Train RF-DETR segmentation from `segpt-train`: the patch cache becomes a COCO folder, an `RFDETRSeg*` model is trained on it, and the run directory looks like any other run.

RF-DETR is an *instance* model (it predicts a set of objects, not a per-pixel map). It lives beside the U-Net models, not inside `get_model()`: `get_model_from_cfg` raises on an unknown `model.source`, so `scripts/train.py` asks `is_rfdetr_cfg(cfg)` first and calls `train_rfdetr` instead of the U-Net trainer.

Config grammar stays the one we already have: `model.source=rfdetr model.arch={nano,small,medium}`. Mapping:

| RF-DETR argument | config key |
|---|---|
| model size | `model.arch` |
| `epochs`, `batch_size`, `num_workers`, `device` | `train.epochs`, `train.batch_size`, `train.num_workers`, `train.device` |
| `lr` | `train.lr` if set, else `train.encoder_lr` |
| `seed` | `train.seed`, else `patch.seed` |
| `num_channels` | channels of `data.data_mode` |
| COCO polygon type | `model.rfdetr.segmentation` (`auto`/`polygon`/`rle`) |

Plan, in order:

1. `is_rfdetr_cfg` and `resolve_rfdetr_cfg`: config in, plain dict out, with clear errors.
1. `prepare_coco`: convert the patch cache once; reuse it when the class names match.
1. `rfdetr_trainer`: the only place that imports `rfdetr`, lazily, with an install hint.
1. `train_rfdetr`: ties them together, takes an injectable trainer factory (tests use a stub, no GPU, no `rfdetr`), writes `checkpoints/last_ema.pth` and `checkpoints/history.json`.
1. `RFDETRPredictor`, `load_rfdetr_predictor`: the other direction. A finished run (`checkpoints/last_ema.pth`) is loaded for prediction, and the instance detections become a `(K, H, W)` score map, one channel per class, so `segpt-eval` and the review report can treat it like a U-Net.

In [ ]:
#| default_exp rfdetr

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import time
from pathlib import Path
from typing import Callable, Optional, Sequence

from omegaconf import DictConfig

import numpy as np
from segmentation_tools_pt.inference import detections_to_semantic
from segmentation_tools_pt.masks import write_coco_folder

## Which config means RF-DETR?

Experiment first on a hand-made config, then promote.

In [ ]:
from omegaconf import OmegaConf
cfg = OmegaConf.create({'model': {'source': 'rfdetr', 'arch': 'small'},
                        'train': {'epochs': 3, 'batch_size': 4, 'encoder_lr': 1e-4, 'decoder_lr': 1e-3,
                                  'num_workers': 0, 'device': 'cuda', 'seed': 7},
                        'patch': {'seed': 42}})
str(cfg.model.get('source', 'smp')).lower() == 'rfdetr', cfg.train.get('lr', cfg.train.encoder_lr)

In [ ]:
#| export
RFDETR_ARCHS = {'nano': 'RFDETRSegNano', 'small': 'RFDETRSegSmall', 'medium': 'RFDETRSegMedium'}
INSTALL_HINT = ("RF-DETR is an optional dependency and is not installed in this environment. "
                "Install it in a separate environment (it needs pre-release packages): "
                "`uv pip install --prerelease=allow 'rfdetr[train]'` "
                "or `uv sync --extra rfdetr --prerelease=allow`.")


def is_rfdetr_cfg(cfg: DictConfig) -> bool:
    """Return True when `model.source` is `rfdetr`, the signal for `scripts/train.py` to skip `get_model_from_cfg`."""
    return str(cfg.model.get('source', 'smp')).lower() == 'rfdetr'


def resolve_rfdetr_cfg(cfg: DictConfig, in_channels: int) -> dict:
    """Map the train config onto RF-DETR arguments and return a plain dict.

    `lr` is `train.lr` when set, else `train.encoder_lr` (RF-DETR has one learning rate, and `decoder_lr` is a U-Net head value).
    Raises `ValueError` for an unknown `model.arch`.
    """
    arch = str(cfg.model.get('arch', 'small')).lower()
    if arch not in RFDETR_ARCHS:
        raise ValueError(f"model.arch={arch!r} is not an RF-DETR size; choose one of {sorted(RFDETR_ARCHS)}")
    tr = cfg.train
    lr = tr.get('lr', None)
    if lr is None:
        lr = tr.get('encoder_lr', 1e-4)
    rf = cfg.model.get('rfdetr', None) or {}
    seed = tr.get('seed', None)
    if seed is None:
        seed = cfg.patch.seed
    return dict(arch=arch, num_channels=int(in_channels), epochs=int(tr.epochs), batch_size=int(tr.batch_size),
                lr=float(lr), seed=int(seed), num_workers=int(tr.get('num_workers', 4)),
                device=str(tr.get('device', 'cuda')), segmentation=str(rf.get('segmentation', 'auto')))

In [ ]:
resolve_rfdetr_cfg(cfg, in_channels=1)

## Patch cache to COCO folder

`write_coco_folder` (from `masks`) does the conversion; layout `auto` finds both the label-map cache and the per-class multilabel cache. Rebuilding costs minutes on a big cache, so an existing folder is reused when its category names match the config.

In [ ]:
#| export
def prepare_coco(patch_dir, coco_dir, class_names: Sequence[str], segmentation: str = 'auto') -> Path:
    """Convert the patch cache to a COCO folder at `coco_dir` (skipped when one with the same class names exists) and return it."""
    coco = Path(coco_dir)
    ann = coco / 'valid' / '_annotations.coco.json'
    names = list(class_names)
    if ann.exists():
        have = [c['name'] for c in sorted(json.loads(ann.read_text())['categories'], key=lambda c: c['id'])]
        if have == names:
            return coco
    summary = write_coco_folder(str(patch_dir), str(coco), names, layout='auto', segmentation=segmentation)
    print(f'COCO folder {coco}: {summary}')
    return coco

## The only place that imports `rfdetr`

In [ ]:
#| export
def rfdetr_trainer(arch: str, num_channels: int):
    """Return an `RFDETRSeg*` model for `arch` (nano|small|medium); raises `ImportError(INSTALL_HINT)` when `rfdetr` is missing."""
    try:
        import rfdetr
    except ImportError as e:
        raise ImportError(INSTALL_HINT) from e
    return getattr(rfdetr, RFDETR_ARCHS[arch])(num_channels=num_channels)

## Train, and write the usual run layout

Inside the Hydra run directory (`run_dir`, the cwd): `.hydra/config.yaml` is written by Hydra; here we add `coco/` and `checkpoints/{last_ema.pth, history.json}`. `last_ema.pth` is the checkpoint to evaluate (RF-DETR's `best_total` can pick epoch 1, see #106).

In [ ]:
#| export
def train_rfdetr(cfg: DictConfig, in_channels: int, patch_dir, class_names: Optional[Sequence[str]] = None,
                 run_dir='.', trainer_factory: Optional[Callable] = None) -> dict:
    """Train RF-DETR segmentation on the patch cache and return the run summary (also saved as `checkpoints/history.json`).

    `trainer_factory(arch, num_channels)` must return an object with a `train(**kwargs)` method; the default is `rfdetr_trainer`.
    `class_names=None` means one class called `defect` (a binary mask cache).
    """
    p = resolve_rfdetr_cfg(cfg, in_channels)
    names = list(class_names) if class_names else ['defect']
    run_dir = Path(run_dir)
    ckpt = run_dir / 'checkpoints'
    ckpt.mkdir(parents=True, exist_ok=True)
    coco = prepare_coco(patch_dir, run_dir / 'coco', names, p['segmentation'])
    model = (trainer_factory or rfdetr_trainer)(p['arch'], p['num_channels'])
    t0 = time.time()
    model.train(dataset_dir=str(coco), epochs=p['epochs'], batch_size=p['batch_size'], device=p['device'],
                num_workers=p['num_workers'], output_dir=str(ckpt), lr=p['lr'], early_stopping=False, seed=p['seed'])
    last = ckpt / 'last_ema.pth'
    if not last.exists():
        raise FileNotFoundError(f'RF-DETR finished but {last} was not written')
    hist = dict(model=f"rfdetr_{p['arch']}", class_names=names, coco_dir=str(coco),
                train_seconds=round(time.time() - t0), checkpoint=str(last), **{k: p[k] for k in
                ('epochs', 'batch_size', 'lr', 'seed', 'num_channels')})
    (ckpt / 'history.json').write_text(json.dumps(hist, indent=1))
    return hist

## Predict with a finished run

Training wrote `checkpoints/last_ema.pth`. To score it like a U-Net we need a per-class pixel map, not a list of objects. RF-DETR returns detections (`mask (N,H,W)`, `class_id (N,)`, `confidence (N,)`, 0-based class ids); `detections_to_semantic(first_class_id=0, output='stack')` paints them into `(K, H, W)` where each pixel holds the best score of that class, and `threshold` drops weak detections.

Two thresholds are in play, so keep them apart: RF-DETR is asked **once** for everything above the lowest swept score (`detections(image, floor)`), and each sweep point is then cut from the same detections (`stack(dets, shape, t)`). Cutting is identical to asking at `t`, but costs a rasterisation instead of a forward pass.

RF-DETR scores are not calibrated like U-Net probabilities: the best threshold of the #106 runs sat at 0.1 to 0.3, so its default grid is denser at the low end than the U-Net one.

In [ ]:
#| export
RFDETR_THRESHOLDS = (0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7)


def rfdetr_checkpoint(run_dir) -> Path:
    "Return `<run_dir>/checkpoints/last_ema.pth`, the checkpoint to evaluate; raises `FileNotFoundError` when it is missing."
    p = Path(run_dir) / 'checkpoints' / 'last_ema.pth'
    if not p.exists():
        raise FileNotFoundError(f'{p} not found: {run_dir} has no RF-DETR checkpoint (train with model.source=rfdetr)')
    return p


def rfdetr_loader(ckpt):
    "Load a trained `RFDETR` model from `ckpt`; the only loading call that imports `rfdetr` (raises `ImportError(INSTALL_HINT)` without it)."
    try:
        from rfdetr import RFDETR
    except ImportError as e:
        raise ImportError(INSTALL_HINT) from e
    return RFDETR.from_checkpoint(str(ckpt))


class RFDETRPredictor:
    """A loaded RF-DETR model plus the rule that turns its detections into a `(num_classes, H, W)` score map.

    `model` needs `predict(image, threshold=...)` returning `supervision.Detections`-like objects.
    """
    def __init__(self, model, num_classes: int):
        self.model, self.num_classes = model, int(num_classes)

    def detections(self, image, threshold: float = 0.0):
        "RF-DETR detections of one image (an `(H, W[, C])` array or a PIL image) scoring at least `threshold`."
        if isinstance(image, np.ndarray):
            from PIL import Image
            image = Image.fromarray(image)
        return self.model.predict(image, threshold=threshold)

    def stack(self, dets, shape, threshold: float = 0.0):
        "Rasterise `dets` to `(num_classes, H, W)` float32: best score per class and pixel among detections scoring at least `threshold`."
        return detections_to_semantic(dets, shape=tuple(shape), class_ids=range(self.num_classes), first_class_id=0,
                                      output='stack', threshold=threshold)

    def __call__(self, image, threshold: float = 0.5):
        "Score map of one image at `threshold`; `> 0` is the binary prediction."
        return self.stack(self.detections(image, threshold), np.asarray(image).shape[:2], threshold)


def load_rfdetr_predictor(run_dir, num_classes: Optional[int] = None, loader: Optional[Callable] = None) -> RFDETRPredictor:
    """Load `checkpoints/last_ema.pth` of an RF-DETR run as an `RFDETRPredictor`.

    `num_classes` defaults to the number of `data.class_names` in the run's `.hydra/config.yaml` (1 when there are none).
    `loader(ckpt)` returns the model; the default is `rfdetr_loader` (tests pass a stub).
    """
    ckpt = rfdetr_checkpoint(run_dir)
    if num_classes is None:
        from omegaconf import OmegaConf
        cfg_path = Path(run_dir) / '.hydra' / 'config.yaml'
        names = OmegaConf.load(cfg_path).data.get('class_names', None) if cfg_path.exists() else None
        num_classes = len(names) if names else 1
    return RFDETRPredictor((loader or rfdetr_loader)(ckpt), num_classes)


## Tests

A stub trainer stands in for `rfdetr`: it records the arguments and writes `last_ema.pth`, so none of this needs a GPU.

In [ ]:
#| test
import cv2, numpy as np, tempfile
from omegaconf import OmegaConf

def _raises(exc, match, fn, *a, **k):
    try: fn(*a, **k)
    except exc as e:
        assert match in str(e), str(e); return
    raise AssertionError(f'{exc.__name__} not raised')

def _cache(root, names):
    rng = np.random.default_rng(0)
    for split, n in (('train', 3), ('val', 2)):
        (root / split / 'images').mkdir(parents=True); (root / split / 'masks').mkdir(parents=True)
        for i in range(n):
            cv2.imwrite(str(root / split / 'images' / f'p{i}.png'), rng.integers(0, 255, (32, 32), dtype=np.uint8))
            for k, c in enumerate(names):
                m = np.zeros((32, 32), np.uint8); m[4 + 8 * k:12 + 8 * k, 4:20] = 255
                cv2.imwrite(str(root / split / 'masks' / f'p{i}_{c}.png'), m)

class _Stub:
    calls = []
    def __init__(self, arch, ch): self.arch, self.ch = arch, ch
    def train(self, **kw):
        _Stub.calls.append((self.arch, self.ch, kw))
        (Path(kw['output_dir']) / 'last_ema.pth').write_bytes(b'x')

cfg = OmegaConf.create({'model': {'source': 'rfdetr', 'arch': 'nano', 'rfdetr': {'segmentation': 'polygon'}},
                        'train': {'epochs': 2, 'batch_size': 3, 'encoder_lr': 5e-5, 'decoder_lr': 1e-3,
                                  'num_workers': 0, 'device': 'cpu', 'seed': 9}, 'patch': {'seed': 1}})
assert is_rfdetr_cfg(cfg) and not is_rfdetr_cfg(OmegaConf.create({'model': {'source': 'smp'}}))
assert not is_rfdetr_cfg(OmegaConf.create({'model': {}}))
r = resolve_rfdetr_cfg(cfg, 1)
assert (r['arch'], r['lr'], r['seed'], r['num_channels'], r['segmentation']) == ('nano', 5e-5, 9, 1, 'polygon')
cfg_lr = OmegaConf.merge(cfg, {'train': {'lr': 2e-4}})
assert resolve_rfdetr_cfg(cfg_lr, 1)['lr'] == 2e-4
_raises(ValueError, 'RF-DETR size', resolve_rfdetr_cfg, OmegaConf.merge(cfg, {'model': {'arch': 'resnet18'}}), 1)

with tempfile.TemporaryDirectory() as d:
    d = Path(d); names = ['chip', 'lifted']; _cache(d / 'cache', names)
    h = train_rfdetr(cfg, 1, d / 'cache', names, run_dir=d / 'run', trainer_factory=_Stub)
    arch, ch, kw = _Stub.calls[-1]
    assert (arch, ch) == ('nano', 1)
    assert (kw['epochs'], kw['batch_size'], kw['lr'], kw['seed'], kw['device']) == (2, 3, 5e-5, 9, 'cpu')
    assert kw['early_stopping'] is False and kw['output_dir'] == str(d / 'run' / 'checkpoints')
    assert (d / 'run' / 'checkpoints' / 'last_ema.pth').exists()
    assert json.loads((d / 'run' / 'checkpoints' / 'history.json').read_text())['model'] == 'rfdetr_nano'
    ann = json.loads((d / 'run' / 'coco' / 'train' / '_annotations.coco.json').read_text())
    assert [c['name'] for c in ann['categories']] == names and len(ann['annotations']) == 6
    mt = (d / 'run' / 'coco' / 'valid' / '_annotations.coco.json').stat().st_mtime_ns
    train_rfdetr(cfg, 1, d / 'cache', names, run_dir=d / 'run', trainer_factory=_Stub)
    assert (d / 'run' / 'coco' / 'valid' / '_annotations.coco.json').stat().st_mtime_ns == mt   # reused, not rebuilt
    train_rfdetr(cfg, 1, d / 'cache', ['chip'], run_dir=d / 'run', trainer_factory=_Stub)       # class names changed -> rebuilt
    assert len(json.loads((d / 'run' / 'coco' / 'valid' / '_annotations.coco.json').read_text())['categories']) == 1

    class _NoCkpt(_Stub):
        def train(self, **kw): pass
    _raises(FileNotFoundError, 'last_ema.pth', train_rfdetr, cfg, 1, d / 'cache', names, run_dir=d / 'run2', trainer_factory=_NoCkpt)

In [ ]:
#| test
import importlib.util
if importlib.util.find_spec('rfdetr') is None:
    _raises(ImportError, 'prerelease=allow', rfdetr_trainer, 'small', 1)

In [ ]:
#| test
import numpy as np
from types import SimpleNamespace

def _dets(*items, shape=(8, 8)):
    "Fake `supervision.Detections`: items are (class_id, confidence, rows, cols)."
    m = np.zeros((len(items),) + shape, bool)
    for k, (_, _, r, c) in enumerate(items):
        m[k, r, c] = True
    return SimpleNamespace(mask=m, class_id=np.array([x[0] for x in items], int), confidence=np.array([x[1] for x in items], float))

class _FakeModel:
    "Stands in for `RFDETR`: records the threshold it was asked for and drops detections below it."
    def __init__(self): self.calls = []
    def predict(self, img, threshold=0.5):
        self.calls.append((np.asarray(img).shape, threshold))
        all_ = [(0, 0.9, slice(0, 2), slice(0, 2)), (1, 0.4, slice(4, 6), slice(4, 6)), (1, 0.05, slice(6, 8), slice(0, 2))]
        return _dets(*[x for x in all_ if x[1] >= threshold])

fm = _FakeModel()
pred = RFDETRPredictor(fm, 2)
img = np.zeros((8, 8), np.uint8)
s_hi = pred(img, threshold=0.5)
assert s_hi.shape == (2, 8, 8) and (s_hi[0] > 0).sum() == 4 and (s_hi[1] > 0).sum() == 0       # class 1 det scores 0.4 < 0.5
s_lo = pred(img, threshold=0.3)
assert (s_lo[1] > 0).sum() == 4 and abs(s_lo[1].max() - 0.4) < 1e-6                              # 0-based ids: class_id 1 is channel 1
assert fm.calls[-1][1] == 0.3
# one prediction at the floor, many thresholds: same as predicting at each threshold
dets = pred.detections(img, 0.0)
for t in (0.03, 0.3, 0.5):
    assert np.array_equal(pred.stack(dets, (8, 8), t) > 0, pred(img, threshold=t) > 0)
empty = pred.stack(_dets(shape=(8, 8)), (8, 8), 0.5)
assert empty.shape == (2, 8, 8) and not empty.any()
assert RFDETR_THRESHOLDS == (0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7)


In [ ]:
#| test
with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    _raises(FileNotFoundError, 'last_ema.pth', rfdetr_checkpoint, d)
    (d / 'checkpoints').mkdir(); (d / 'checkpoints' / 'last_ema.pth').write_bytes(b'x')
    assert rfdetr_checkpoint(d) == d / 'checkpoints' / 'last_ema.pth'
    seen = []
    p = load_rfdetr_predictor(d, num_classes=2, loader=lambda ckpt: seen.append(ckpt) or _FakeModel())
    assert seen == [d / 'checkpoints' / 'last_ema.pth'] and isinstance(p, RFDETRPredictor) and p.num_classes == 2
    cfgd = d / '.hydra'; cfgd.mkdir()
    (cfgd / 'config.yaml').write_text('data: {class_names: [a, b, c]}\nmodel: {source: rfdetr, arch: small}\n')
    assert load_rfdetr_predictor(d, loader=lambda c: _FakeModel()).num_classes == 3      # classes read from the run config
if importlib.util.find_spec('rfdetr') is None:
    _raises(ImportError, 'prerelease=allow', rfdetr_loader, Path('x.pth'))


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()